In [ ]:
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

---

## Vorverarbeitung der Daten

In [ ]:
df = pd.read_csv("data.csv")
df["date"] = pd.to_datetime(df["DateTime"])

# Tarife werden vorerst zusammengelegt, "date" wird dabei als Index übernommen
df = df.groupby("date").agg({"KWH": "sum", "meters": "sum"})

# Normalisierung: Stromverbrauch eines Verbrauchers für jeden Zeitpunkt
df["kwh"] = df["KWH"] / df["meters"]

df = df[["KWH", "kwh", "meters"]]

# Durch .asfreq() fallen die fehlerhaften Datenpunkte weg
df = df.sort_index().asfreq("30min")

---

## Untersuchung der Zeitreihe auf Trend, Saisonalität und Residuum

In [ ]:
def analyse_trend(df: pd.DataFrame) -> pd.DataFrame:
    # Berechnung des linearen Trends der Zeitreihe
    x = mdates.date2num(df.index)
    p = np.poly1d(np.polyfit(x, df["kwh"], 1))
    df["kwh_trend"] = p(x)

    # Der Stromverbrauch bereinigt vom Trend
    df["kwh_trendbereinigt"] = df["kwh"] - df["kwh_trend"]

    return df


def analyse_seasonality(df: pd.DataFrame) -> pd.DataFrame:
    df["focus"] = df.index.hour * 2 + (df.index.minute // 30)  # Fokus auf einen Tag setzen

    # Der Mittelwert aller 30-Minuten Intervalle eines Tages
    seasonal = df.groupby("focus")["kwh_trendbereinigt"].mean()

    # .transform berechnet den Mittelwert pro Gruppe und weist ihn jedem Datenpunkt der Gruppe zu
    df["kwh_trendbereinigt_saison"] = df.groupby("focus")["kwh_trendbereinigt"].transform("mean")
    return df.drop(columns=["focus"])


df = analyse_trend(df)
df = analyse_seasonality(df)

In [ ]:
def analyse_cleaned(df: pd.DataFrame) -> pd.DataFrame:
    # Berechnung der bereinigte Zeitreihe bzw. dem Residuum/Rest
    df["kwh_bereinigt"] = df["kwh_trendbereinigt"] - df["kwh_trendbereinigt_saison"]

    # Getrennte Plots
    _, ax = plt.subplots(4, 1, figsize=(15, 6), sharex=True)

    for p in ax:
        p.set_xlim(pd.Series(df.index).quantile(0.46), pd.Series(df.index).quantile(0.52))

    ax[0].plot(df["kwh"], color="#6B001E")
    ax[0].set_title("Stromverbrauch")

    ax[1].plot(df["kwh_trend"], color="#000000FF")
    ax[1].set_title("Trend")

    ax[2].plot(df["kwh_trendbereinigt_saison"], color="#D386FF")
    ax[2].set_title("Saisonalität")

    ax[3].plot(df["kwh_bereinigt"], color="#9B0000")
    ax[3].set_title("Residuum")

    plt.tight_layout()
    plt.close()

    # Residuum der gesamten Zeitreihe
    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(df.index, df["kwh_bereinigt"], color="#9B0000", alpha=0.9, label="Verbrauch (Bereinigt/Residuum)")
    ax.set(
        title="Stromverbrauch eines Verbrauchers",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()
    return df


df = analyse_cleaned(df)

---

## STL / Autokorrelation und Partielle Autokorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import MSTL


def plot_STL(df: pd.DataFrame, periods: tuple) -> pd.DataFrame:
    decomposition = MSTL(df["kwh"], periods=periods)
    decomposition = decomposition.fit()

    _, ax = plt.subplots(4, 1, figsize=(15, 6), sharex=True)

    for p in ax:
        p.set_xlim(pd.Series(df.index).quantile(0.10), pd.Series(df.index).quantile(0.21))

    ax[0].plot(df["kwh"], color="#6B001E")
    ax[0].set_title("Stromverbrauch")

    ax[1].plot(decomposition.trend, color="#000000FF")
    ax[1].set_title("Trend")

    ax[2].plot(decomposition.seasonal, color="#D386FF")
    ax[2].set_title("Saisonalität")

    ax[3].plot(decomposition.resid, color="#9B0000")
    ax[3].set_title("Residuum")

    plt.tight_layout()
    plt.close()

    # Residuum der gesamten Zeitreihe
    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(df.index, decomposition.resid, color="#9B0000", alpha=0.9, label="Verbrauch (Bereinigt/Residuum)")
    ax.set(
        title="Stromverbrauch eines Verbrauchers",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()

    return decomposition

In [ ]:
decomposition = plot_STL(df, (48))

_, ax = plt.subplots(1, 2, figsize=(15, 6))

plot_acf(decomposition.resid, lags=48, ax=ax[0])
ax[0].set_title("Autokorrelation")

plot_pacf(decomposition.resid, lags=48, ax=ax[1])
ax[1].set_title("Partielle Autokorrelation")

plt.tight_layout()
plt.show()

# ACF vom Resid zeigt noch Restmuster (Wellenform, negative Lags (-> daher Saisonalität; Falls ohne Vorzeichenwechsel -> nicht bereinigter Trend?))
# Jedenfalls Hinweis auf nicht erfasste Wochensaisonalität, siehe zweites ACF. Antwort formulieren in Markdown-Block.
# Vielleicht beim eigenen Ansatz oben auch rausrechnen.

In [ ]:
decomposition = plot_STL(df, (48, 336))  # Monat? Macht keinen Sinn; Jahr? Da nur zwei im Datensatz, zu wenig?

_, ax = plt.subplots(1, 2, figsize=(15, 6))

plot_acf(decomposition.resid, lags=48, ax=ax[0])
ax[0].set_title("Autokorrelation")

plot_pacf(decomposition.resid, lags=48, ax=ax[1])
ax[1].set_title("Partielle Autokorrelation")

plt.tight_layout()
plt.show()

In [ ]:
# Vorschlag für Aufgabe 2:
# Ein Modell entfernen um Code zu kürzen.
# Beantwortung der Aufgabe mit nicht entfernter Saisonalität.
# LR mit MSTL versuchen, mit Hinweis auf Data Leakage (vernachlässigbar, da nur 6h?).

---

## Vorhersage mit Linearer Regression

Zielvariable hier ist zunächst der tatsächliche Verbrauch 6 Stunden in der Zukunft.

Da STL für seine Decompesition auch Daten aus der Zukunft nimmt, würde die Weiterverwendung für das Training von Modellen Data Leakage einführen.

Deshalb werden für die Trainingsdaten wieder auf die manuelle Zerlegung zurückgegriffen.

In [ ]:
# Funktionen für die Modellanalyse
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error


def plot_pred(test: pd.DataFrame, pred: np.ndarray, col: str) -> None:
    _, ax = plt.subplots(figsize=(15, 6))

    ax.plot(test.index, test[col], color="#6B001E", alpha=0.7, label="Tatsächlicher Verbrauch")
    ax.plot(test.index, pred, color="#0029BC", alpha=1, label="Prognose")

    ax.set(
        title="Prognose 6 Stunden in die Zukunft",
        xlabel="Datum",
        ylabel="Kilowattstunde",
    )
    ax.set_xlim(pd.Series(test.index).quantile(0.0), pd.Series(test.index).quantile(0.1))
    ax.grid(axis="y", alpha=0.5)
    ax.margins(x=0)
    ax.legend()

    plt.tight_layout()
    plt.show()


def show_metrics(y_test: np.ndarray, pred: np.ndarray, model: LinearRegression, verbose: bool = True) -> tuple[float, float, float, float]:
    r2 = r2_score(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = root_mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)

    if verbose:
        print(
            f"Regression Intercept: {model.intercept_}\n"
            f"Regression Coefficients: {model.coef_}\n"
            f"R²:\t{r2: .6f}\n"
            f"MSE:\t{mse: .6f}\n"
            f"RMSE:\t{rmse: .6f}\n"
            f"MAE:\t{mae: .6f}\n"
        )

    return r2, mse, rmse, mae

In [ ]:
df_model_a = df.copy()
# Aufteilung der Trainings- und Testdaten
split = int(len(df_model_a) * 0.7)
train = df_model_a.iloc[:split]
test = df_model_a.iloc[split:]

In [ ]:
# Trend
x_train = mdates.date2num(train.index)
p = np.poly1d(np.polyfit(x_train, train["kwh"], 1))

train["kwh_trend"] = p(mdates.date2num(train.index))
test["kwh_trend"] = p(mdates.date2num(test.index))  # Extrapolation auf Testdaten

# Trendbereinigung
train["kwh_trendbereinigt"] = train["kwh"] - train["kwh_trend"]
test["kwh_trendbereinigt"] = test["kwh"] - test["kwh_trend"]


# Saisonalität
train["focus"] = train.index.hour * 2 + (train.index.minute // 30)
seasonal = train.groupby("focus")["kwh_trendbereinigt"].mean()  # Nur Train

test["focus"] = test.index.hour * 2 + (test.index.minute // 30)

train["kwh_saison"] = train["focus"].map(seasonal)
test["kwh_saison"] = test["focus"].map(seasonal)  # Mapping, kein Refit


# Residuum
train["kwh_bereinigt"] = train["kwh_trendbereinigt"] - train["kwh_saison"]
test["kwh_bereinigt"] = test["kwh_trendbereinigt"] - test["kwh_saison"]


# Aufteilung der Spalten nach Train und Test
for split_df in [train, test]:
    split_df["kwh_bereinigt_t-1"] = split_df["kwh_bereinigt"].shift(1)
    split_df["kwh_saison_t+12"] = split_df["kwh_saison"].shift(-12)
    split_df["kwh_target"] = split_df["kwh"].shift(-12)

last_row = test.iloc[[-1]].copy()

# Zukunfstwert für die letzte Zeile berechnen
future_time = last_row.index[0] + pd.Timedelta(hours=6)
future_focus = future_time.hour * 2 + (future_time.minute // 30)

last_row["kwh_trend_t+12"] = p(mdates.date2num([future_time]))[0]
last_row["kwh_saison_t+12"] = seasonal.loc[future_focus]

train = train.dropna()
test = test.dropna()
df_model_a = pd.concat([test, last_row])

#### Hinweis zum Shift der Saisonalität:
Die Saison berechnet sich aus den Tagesmittelwerten.
Dem Modell wird durch den Shift mitgegeben wie sich der Zyklus zum Zeitpunkt seiner geplanten Vorhersage ändert.

Würde der aktuelle Zeitpunkt für den Zyklus gewählt werden, würde das Modell z.B. den Morgenzyklus auf den Mittagswert anwenden.

In [ ]:
# Trainieren des Models
features_a = ["kwh_bereinigt", "kwh_bereinigt_t-1", "kwh_trend", "kwh_saison_t+12"]

X_train_a = train[features_a]
y_train_a = train["kwh_target"]  # Den um 6 Stunden verschobenen Wert nehmen, nicht den Original Wert
model_a = LinearRegression()
model_a.fit(X_train_a, y_train_a)

# Vorhersage auf Testdaten
X_test_a = test[features_a]
y_pred_a = model_a.predict(X_test_a)

plot_pred(test, y_pred_a, "kwh_target")
_ = show_metrics(test["kwh_target"], y_pred_a, model_a)

In [ ]:
test["kwh_target"]

<font color="orange">
Erkenntnisse: Das Modell erkennt Saisonalität und Trend gut, hat aber Probleme die Varianz des Residuums genauer einzugrenzen.

Insgesamt liefert das Modell aber gute Prognosen, die nahe der echten Daten liegen.
</font>

---

Als Zielvariable wird nun das Residuum 6 Stunden in der Zukunft verwendet.

Für die eigentliche Vorhersage werden der zuvor berechnete Trend und Saisonalität wieder einberechnet. 

In [ ]:
df_model_b = df.copy()
# Train-Test-Split
split = int(len(df_model_b) * 0.7)
train = df_model_b.iloc[:split].copy()
test = df_model_b.iloc[split:].copy()

In [ ]:
# Trend
x_train = mdates.date2num(train.index)
p = np.poly1d(np.polyfit(x_train, train["kwh"], 1))

train["kwh_trend"] = p(mdates.date2num(train.index))
test["kwh_trend"] = p(mdates.date2num(test.index))  # Extrapolation

# Trendbereinigung
train["kwh_trendbereinigt"] = train["kwh"] - train["kwh_trend"]
test["kwh_trendbereinigt"] = test["kwh"] - test["kwh_trend"]


# Saisonalität
train["focus"] = train.index.hour * 2 + (train.index.minute // 30)
seasonal = train.groupby("focus")["kwh_trendbereinigt"].mean()

test["focus"] = test.index.hour * 2 + (test.index.minute // 30)

train["kwh_saison"] = train["focus"].map(seasonal)
test["kwh_saison"] = test["focus"].map(seasonal)


# Residuum
train["kwh_bereinigt"] = train["kwh_trendbereinigt"] - train["kwh_saison"]
test["kwh_bereinigt"] = test["kwh_trendbereinigt"] - test["kwh_saison"]


# Aufteilung der Spalten nach Train und Test
for split_df in [train, test]:
    split_df["kwh_bereinigt_t-1"] = split_df["kwh_bereinigt"].shift(1)

    # Trend und Saison bei t+12 – nur für Rekomposition, nicht als Features
    split_df["kwh_trend_t+12"] = split_df["kwh_trend"].shift(-12)
    split_df["kwh_saison_t+12"] = split_df["kwh_saison"].shift(-12)
    # Zielwerte
    split_df["kwh_target_bereinigt"] = split_df["kwh_bereinigt"].shift(-12)
    split_df["kwh_target"] = split_df["kwh"].shift(-12)

last_row = test.iloc[[-1]].copy()


future_time = last_row.index[0] + pd.Timedelta(hours=6)
future_focus = future_time.hour * 2 + (future_time.minute // 30)

last_row["kwh_trend_t+12"] = p(mdates.date2num([future_time]))[0]
last_row["kwh_saison_t+12"] = seasonal.loc[future_focus]

train = train.dropna()
test = test.dropna()

df_model_b = pd.concat([test, last_row])

In [ ]:
features_b = ["kwh_bereinigt", "kwh_bereinigt_t-1"]
# Training (nur Residuum-Lags als Features)
X_train = train[features_b]
y_train = train["kwh_target_bereinigt"]

model_b = LinearRegression()
model_b.fit(X_train, y_train)

# Vorhersage der Testdaten
X_test_b = test[features_b]
y_pred_residuum = model_b.predict(X_test_b)  # Nur das Residuum wird hier vorhergesagt

# Trend und Saisonalität bei t+12 liegen durch .shift(-12) bereits im test-Index vor
y_pred_b = y_pred_residuum + test["kwh_trend_t+12"] + test["kwh_saison_t+12"]

plot_pred(test, y_pred_b, "kwh_target")
_ = show_metrics(test["kwh_target"], y_pred_b, model_b)

<font color="orange">
Erkenntnisse: Das Modell schätzt auch ohne mitgegebenen Trend und Saisonalität das Residuum beinahe exakt gleich ein.

Ein Grund dafür ist durch die PACF gezeigte Abhängigkeit von nur einen Lag. Bei 12 Lags in die Zukunft geht die Abhängigkeit gegen 0 und ist nur noch unerklärbares Rauschen.

Eine Möglichkeit für bessere Ergebnisse wäre z.B. eine feinere Saisonalitätsbestimmung (Unterscheidung zwischen Wochentagen und Wochenenden).
</font>

In [ ]:
# Letzter bekannter Datenpunkt für Zukunftsprognose
last_row_a = df_model_a.iloc[[-1]]
last_row_b = df_model_b.iloc[[-1]]

X_future_a = last_row[features_a]
X_future_b = last_row[features_b]

# Zukunftsprognose Option A
future_pred_a = model_a.predict(X_future_a)[0]
future_time = df_model_a.index[-1] + pd.Timedelta(hours=6)

# Zukunftsprognose Option B
future_residuum_b = model_b.predict(X_future_b)[0]
future_pred_b = future_residuum_b + last_row_b["kwh_trend_t+12"].values[0] + last_row_b["kwh_saison_t+12"].values[0]

_, ax = plt.subplots(figsize=(15, 7))

ax.plot(
    test.index,
    test["kwh_target"],
    color="#6B001E",
    alpha=0.7,
    linewidth=2,
    label="Tatsächlicher Verbrauch (t+12)",
)
ax.plot(test.index, y_pred_a, color="#C45E00", alpha=0.6, linewidth=2, label="Option A (direkt auf kwh)")
ax.plot(test.index, y_pred_b, color="#1A8F3D", alpha=0.6, linewidth=2, label="Option B (Residuum + Trend + Saison)")

ax.scatter(future_time, future_pred_a, color="#C45E00", s=80, zorder=5, label=f"Option A: +6h = {future_pred_a:.4f} kWh")
ax.scatter(future_time, future_pred_b, color="#1A8F3D", s=80, zorder=5, label=f"Option B: +6h = {future_pred_b:.4f} kWh")
ax.axvline(x=test.index[-1], color="gray", linestyle="--", alpha=1, label="Ende Testdaten")

ax.set(
    title="Vergleich von Option A und Option B",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.set_xlim(pd.Series(test.index).quantile(0.95), future_time + pd.Timedelta(hours=1))
ax.grid(axis="y", alpha=0.5)
ax.margins(x=0)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

r2_a, mse_a, rmse_a, mae_a = show_metrics(test["kwh_target"], y_pred_a, model_a, verbose=False)
r2_b, mse_b, rmse_b, mae_b = show_metrics(test["kwh_target"], y_pred_b, model_b, verbose=False)

# Metriken
r2_diff = (r2_b - r2_a) / abs(r2_a) * 100
rmse_diff = (rmse_a - rmse_b) / rmse_a * 100
mae_diff = (mae_a - mae_b) / mae_a * 100

print(
    f"{'Metrik':<8} {'Option A':>10} {'Option B':>10} {'Verbesserung':>14}\n"
    f"{'-' * 45}\n"
    f"{'R²':<8} {r2_a:>10.6f} {r2_b:>10.6f} {r2_diff:>+13.4f}%\n"
    f"{'RMSE':<8} {rmse_a:>10.6f} {rmse_b:>10.6f} {rmse_diff:>+13.4f}%\n"
    f"{'MAE':<8} {mae_a:>10.6f} {mae_b:>10.6f} {mae_diff:>+13.4f}%\n"
)

## Varianzen

In [ ]:
# Berechnung der Varianzen
def variances(df: pd.DataFrame):
    df_var = df.copy()

    df_var["trend"] = df_var["kwh_trend"]
    df_var["seasonal"] = df_var["kwh_trendbereinigt_saison"]
    df_var["resid"] = df_var["kwh_bereinigt"]

    variances = df_var[["kwh", "trend", "seasonal", "resid"]].var()
    variances["ryth"] = df_var["seasonal"].var()
    portions = variances / variances["kwh"]

    print(
        f"Von der Varianz der Zeitreihe werden:\n\t{(portions['trend'] * 100):2.1f}% durch den Trend erklärt"
        f"\n\t{(portions['ryth'] * 100):2.1f}% durch die Periodizität erklärt"
        f"\n\t{(portions['resid'] * 100):2.1f}% haben andere Ursachen (Residuum)"
    )


variances(df)